# Customer Support RAG Engine & Automated LLM Evaluation Benchmark

This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) system for customer service policy retrieval, paired with an automated **LLM-as-a-Judge** framework to evaluate Faithfulness, Answer Relevance, and Instruction Compliance against curated Ground Truth test cases.

In [2]:
# Install required libraries
!pip install -q langchain langchain-community langchain-groq chromadb langchain-huggingface sentence-transformers pandas tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7

In [3]:
import os
import re
import logging
import warnings
import getpass
import pandas as pd
from tabulate import tabulate

warnings.filterwarnings("ignore")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("All dependencies and modules imported successfully.")

All dependencies and modules imported successfully.


## 1. Model & API Configuration
We configure Groq Cloud endpoints utilizing **Qwen 2.5 (27B)** for generation and **GPT-OSS (120B)** as the evaluation judge.

In [4]:
# Configure Groq API key securely
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

# Generator & Judge Models
GENERATOR_MODEL = "qwen/qwen3.6-27b"
JUDGE_MODEL = "openai/gpt-oss-120b"

print(f"--> Generator Model Assigned : {GENERATOR_MODEL}")
print(f"--> Evaluation Judge Model   : {JUDGE_MODEL}")

Enter Groq API Key: ··········
--> Generator Model Assigned : qwen/qwen3.6-27b
--> Evaluation Judge Model   : openai/gpt-oss-120b


## 2. Knowledge Base Ingestion
We define the official Standard Operating Procedures (SOP) dataset for digital payment refunds, KYC verification, and transaction incident management.

In [5]:
knowledge_base_text = """
# Digital Pay & E-Commerce Customer Support Operating Procedures

1. Refund Policies & Timelines:
- Refund requests must be submitted within 7 business days of item delivery or transaction processing.
- Transactions made via Virtual Account, Instant Bank Transfer, and QR Code will be refunded within 1 business day (24 hours) directly to the user's in-app wallet balance.
- Credit Card payments require 5 to 14 business days to reflect on the account, depending on the issuing bank.
- Initial transaction and payment processing fees are non-refundable if the cancellation is initiated by the buyer.

2. Account Security & Identity Verification (KYC):
- Users must complete Tier-2 Identity Verification (Government ID upload and biometric selfie scan) for single transactions exceeding $1,000.
- User accounts will be automatically suspended temporarily after 5 consecutive failed PIN or password attempts.
- Account unlocking requires identity verification via a scheduled live video session with an authorized Customer Support specialist.

3. Handling Pending & Failed Transactions:
- If a transaction remains in 'Pending' status for more than 30 minutes, the automated system queries the payment gateway partner.
- If funds are deducted from the user balance but the order fails, an automated system auto-reversal will credit the full amount within 3 business days.
"""

with open("knowledge_base.txt", "w", encoding="utf-8") as f:
    f.write(knowledge_base_text.strip())

print("Knowledge base created successfully: 'knowledge_base.txt'")

Knowledge base created successfully: 'knowledge_base.txt'


## 3. Text Chunking & Dense Vector Indexing (ChromaDB)
The document is partitioned into semantic chunks using `RecursiveCharacterTextSplitter` and embedded into a local Chroma vector database using `all-MiniLM-L6-v2`.

In [6]:
# Load knowledge base document
loader = TextLoader("knowledge_base.txt", encoding="utf-8")
docs = loader.load()

# Split text into semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(docs)

# Initialize local HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Construct vector database
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="customer_support_kb"
)

# Search top-3 relevant context chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Successfully indexed {len(splits)} semantic chunks into ChromaDB.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully indexed 3 semantic chunks into ChromaDB.


## 4. RAG Pipeline Construction & Anti-Hallucination Guardrails
We assemble the LangChain pipeline with strict zero-shot system constraints to prevent model hallucinations on queries outside the retrieved context.

In [7]:
generator_llm = ChatGroq(model_name=GENERATOR_MODEL, temperature=0.0)

rag_template = """You are an official Customer Support AI assistant.
Answer the user's question strictly based ONLY on the provided context below.
If the answer cannot be found in the context, explicitly state: "This information is not available in our official knowledge base."
Do NOT assume, infer, or hallucinate external facts.

Context:
{context}

Question: {question}

Helpful & Accurate Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

def clean_thinking_tags(text: str) -> str:
    """Membersihkan token penalaran internal <think> agar output bersih."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

generator_chain = rag_prompt | generator_llm | StrOutputParser()

print("RAG Generator chain initialized successfully.")

RAG Generator chain initialized successfully.


## 5. Benchmark Test Suite & Ground Truth Formulation
We curate a structured test suite covering Direct Fact Retrieval, Constraint Verification, and Out-of-Domain / Negative Testing.

In [8]:
test_cases = [
    {
        "id": "TC-01",
        "category": "Direct Fact Retrieval",
        "question": "How long does a refund take if paid via QR Code or Virtual Account?",
        "ground_truth": "Within 1 business day (24 hours) directly to the in-app wallet balance."
    },
    {
        "id": "TC-02",
        "category": "Direct Fact Retrieval",
        "question": "How many failed PIN attempts will cause an account to be suspended?",
        "ground_truth": "5 consecutive failed PIN or password attempts."
    },
    {
        "id": "TC-03",
        "category": "Out-of-Domain / Hallucination Test",
        "question": "How can I request a refund for an international flight ticket?",
        "ground_truth": "This information is not available in our official knowledge base."
    },
    {
        "id": "TC-04",
        "category": "Constraint Verification",
        "question": "Are initial processing fees refunded if a customer cancels the order?",
        "ground_truth": "Initial transaction and processing fees are non-refundable if initiated by the buyer."
    }
]

print(f"Defined {len(test_cases)} evaluation test cases.")

Defined 4 evaluation test cases.


## 6. Automated Evaluation Framework (LLM-as-a-Judge)
An objective, high-parameter LLM judges the candidate bot answers across three core metrics:
- **Faithfulness**: Detection of fabricated or hallucinated claims.
- **Answer Relevance**: Directness and conciseness relative to the question.
- **Instruction Compliance**: Strict adherence to boundary constraints.

In [9]:
eval_judge_llm = ChatGroq(model_name=JUDGE_MODEL, temperature=0.0)

judge_template = """You are an expert AI Quality Assurance and Data Evaluation Specialist.
Audit the candidate bot answer against the provided Context and Ground Truth reference.

Context: {context}
User Question: {question}
Ground Truth: {ground_truth}
Bot Answer: {bot_answer}

Provide your evaluation strictly following this key-value format without introductory text:
FAITHFULNESS: [1 if purely grounded in context without external hallucinations, 0 otherwise]
RELEVANCE: [1 if directly answers the intent of the prompt concisely, 0 otherwise]
COMPLIANCE: [1 if adhered to guardrail/out-of-domain boundaries, 0 otherwise]
RATIONALE: [One concise sentence explaining the scores]"""

judge_prompt = ChatPromptTemplate.from_template(judge_template)
judge_chain = judge_prompt | eval_judge_llm | StrOutputParser()

def parse_judge_output(raw_eval: str) -> dict:
    """Mengekstrak feedback LLM Judge menjadi metrik numerik dan alasan kualitatif."""
    try:
        faith = int(re.search(r"FAITHFULNESS:\s*([01])", raw_eval, re.IGNORECASE).group(1))
    except:
        faith = 0
    try:
        relevance = int(re.search(r"RELEVANCE:\s*([01])", raw_eval, re.IGNORECASE).group(1))
    except:
        relevance = 0
    try:
        compliance = int(re.search(r"COMPLIANCE:\s*([01])", raw_eval, re.IGNORECASE).group(1))
    except:
        compliance = 0
    try:
        rationale = re.search(r"RATIONALE:\s*(.*)", raw_eval, re.IGNORECASE).group(1).strip()
    except:
        rationale = raw_eval.strip()

    return {
        "Faithfulness": faith,
        "Relevance": relevance,
        "Compliance": compliance,
        "Rationale": rationale
    }

print("Evaluation Judge pipeline & quantitative parser configured.")

Evaluation Judge pipeline & quantitative parser configured.


## 7. Execution, Metrics Summary, & Dataset Export
Execute the benchmark over all test cases, log retrieved chunks, record model responses, run evaluation scoring, and export the structured results to CSV.

In [10]:
results = []
print("Starting automated benchmark evaluation...\n")

for tc in test_cases:
    print(f"Auditing: {tc['id']} - {tc['category']}")

    # 1. Single Retrieval Step
    retrieved_chunks = retriever.invoke(tc["question"])
    context_text = "\n\n".join(doc.page_content for doc in retrieved_chunks)

    # 2. Generate Output & Clean Artifacts
    raw_bot_response = generator_chain.invoke({
        "context": context_text,
        "question": tc["question"]
    })
    bot_response = clean_thinking_tags(raw_bot_response)

    # 3. LLM-as-a-Judge Audit Step
    raw_feedback = judge_chain.invoke({
        "context": context_text,
        "question": tc["question"],
        "ground_truth": tc["ground_truth"],
        "bot_answer": bot_response
    })

    # 4. Parse Structured Metrics
    metrics = parse_judge_output(raw_feedback)
    is_passed = (metrics["Faithfulness"] == 1 and metrics["Relevance"] == 1 and metrics["Compliance"] == 1)

    results.append({
        "Test ID": tc["id"],
        "Category": tc["category"],
        "Question": tc["question"],
        "Ground Truth": tc["ground_truth"],
        "Bot Output": bot_response,
        "Faithfulness": metrics["Faithfulness"],
        "Answer Relevance": metrics["Relevance"],
        "Instruction Compliance": metrics["Compliance"],
        "Status": "PASS" if is_passed else "FAIL",
        "Judge Rationale": metrics["Rationale"]
    })

# Export logs to CSV
df_results = pd.DataFrame(results)
df_results.to_csv("rag_evaluation_results.csv", index=False)

# Executive Summary Metrics
summary_data = [
    ["Total Test Cases", len(df_results)],
    ["Overall Benchmark Pass Rate", f"{(df_results['Status'] == 'PASS').mean() * 100:.1f}%"],
    ["Mean Faithfulness Score", f"{df_results['Faithfulness'].mean() * 100:.1f}%"],
    ["Mean Answer Relevance Score", f"{df_results['Answer Relevance'].mean() * 100:.1f}%"],
    ["Mean Instruction Compliance Score", f"{df_results['Instruction Compliance'].mean() * 100:.1f}%"],
]

print("\n" + "="*80)
print("BENCHMARK EXECUTIVE SCORECARD")
print("="*80)
print(tabulate(summary_data, headers=["Evaluation Metric", "Benchmark Score"], tablefmt="fancy_grid"))

print("\n" + "="*80)
print("TEST CASE AUDIT LOGS")
print("="*80)
print(tabulate(df_results[["Test ID", "Category", "Status", "Faithfulness", "Answer Relevance", "Instruction Compliance"]], headers="keys", tablefmt="grid"))
print("\nEvaluation logs exported successfully: 'rag_evaluation_results.csv'")

Starting automated benchmark evaluation...

Auditing: TC-01 - Direct Fact Retrieval
Auditing: TC-02 - Direct Fact Retrieval
Auditing: TC-03 - Out-of-Domain / Hallucination Test
Auditing: TC-04 - Constraint Verification

BENCHMARK EXECUTIVE SCORECARD
╒═══════════════════════════════════╤═══════════════════╕
│ Evaluation Metric                 │ Benchmark Score   │
╞═══════════════════════════════════╪═══════════════════╡
│ Total Test Cases                  │ 4                 │
├───────────────────────────────────┼───────────────────┤
│ Overall Benchmark Pass Rate       │ 100.0%            │
├───────────────────────────────────┼───────────────────┤
│ Mean Faithfulness Score           │ 100.0%            │
├───────────────────────────────────┼───────────────────┤
│ Mean Answer Relevance Score       │ 100.0%            │
├───────────────────────────────────┼───────────────────┤
│ Mean Instruction Compliance Score │ 100.0%            │
╘═══════════════════════════════════╧═════════════════

In [11]:
# Inspect detailed Judge scoring feedback for each test case
# Detailed Qualitative Inspection
for i, row in df_results.iterrows():
    print(f"\n==================== {row['Test ID']} : {row['Category']} [{row['Status']}] ====================")
    print(f"Question       : {row['Question']}")
    print(f"Ground Truth   : {row['Ground Truth']}")
    print(f"Bot Output     : {row['Bot Output']}")
    print(f"Scores         : Faithfulness={row['Faithfulness']} | Relevance={row['Answer Relevance']} | Compliance={row['Instruction Compliance']}")
    print(f"Judge Rationale: {row['Judge Rationale']}")


==================== TC-01 : Direct Fact Retrieval [PASS] ====================
Question       : How long does a refund take if paid via QR Code or Virtual Account?
Ground Truth   : Within 1 business day (24 hours) directly to the in-app wallet balance.
Bot Output     : Based on our official procedures, refunds for transactions made via Virtual Account or QR Code will be processed within **1 business day (24 hours)** and credited directly to your in-app wallet balance. 

Please let me know if you need further assistance!
Scores         : Faithfulness=1 | Relevance=1 | Compliance=1
Judge Rationale: The response accurately restates the refund timeline from the context, directly answers the user's question, and stays within allowed content.

==================== TC-02 : Direct Fact Retrieval [PASS] ====================
Question       : How many failed PIN attempts will cause an account to be suspended?
Ground Truth   : 5 consecutive failed PIN or password attempts.
Bot Output     : Based 